# 🎙️ ROTBOTS — The Voice
## Text-to-Speech Narration

---

Generates spoken narration for each scene using Edge-TTS (free).

> **🤔 Think about:** What does a synthetic voice do to trust? Would you believe this narration more or less than a human?

---

In [ ]:
# SETUP
!pip install -q edge-tts
import json, subprocess, asyncio
from pathlib import Path
from IPython.display import display, Markdown, Audio
import edge_tts

from google.colab import drive
drive.mount('/content/drive')
WORK_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
AUDIO_DIR = WORK_DIR / 'audio'; AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Setup complete')

In [ ]:
# LOAD ESSAY + PROMPTS for timing info
essay_file = WORK_DIR / 'essay_script.json'
prompts_file = WORK_DIR / 'prompts.json'

narrations = []
if essay_file.exists():
    with open(essay_file) as f: essay = json.load(f)
    print(f'✅ Loaded essay: {essay.get("title","Untitled")}')
    scene_num = 2  # scene 1 = title card
    for ch in essay.get('chapters', []):
        for sec in ch.get('sections', []):
            text = sec.get('narration', '')
            if text:
                narrations.append({'scene': scene_num, 'narration': text, 'mood': sec.get('mood','')})
                wc = len(text.split())
                print(f'   Scene {scene_num}: {wc} words ≈ {wc/2.5:.0f}s')
            scene_num += 1
    print(f'\n   Total: {len(narrations)} narrations')
else:
    print('❌ No essay_script.json found!')

In [ ]:
# VOICE SELECTION
VOICES = {
    'female_us': 'en-US-AriaNeural', 'male_us': 'en-US-GuyNeural',
    'female_uk': 'en-GB-SoniaNeural', 'male_uk': 'en-GB-RyanNeural',
    'documentary': 'en-US-GuyNeural', 'dramatic': 'en-GB-RyanNeural',
    'casual': 'en-US-AriaNeural', 'energetic': 'en-US-JennyNeural',
}

CHOSEN_VOICE = 'documentary'  # Options: female_us, male_us, female_uk, male_uk, documentary, dramatic, casual, energetic
voice_name = VOICES[CHOSEN_VOICE]
print(f'🎙️ Voice: {CHOSEN_VOICE} ({voice_name})')

In [ ]:
# GENERATE NARRATION AUDIO

async def generate_tts(text, path, voice, rate='+0%'):
    comm = edge_tts.Communicate(text, voice, rate=rate)
    await comm.save(str(path))

def get_duration(path):
    try:
        r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)], capture_output=True, text=True, timeout=10)
        return float(r.stdout.strip())
    except: return 5.0

print(f'🎙️ Generating {len(narrations)} narrations...')
audio_files = []
for n in narrations:
    out = AUDIO_DIR / f'narration_{n["scene"]:03d}.mp3'
    wc = len(n['narration'].split())
    print(f'   Scene {n["scene"]}: {wc} words...', end=' ', flush=True)
    try:
        await generate_tts(n['narration'], out, voice_name)
        dur = get_duration(out)
        audio_files.append({'scene': n['scene'], 'path': str(out), 'duration': dur, 'text': n['narration']})
        print(f'✅ {dur:.1f}s')
    except Exception as e: print(f'❌ {e}')

# Save manifest
with open(WORK_DIR / 'audio_manifest.json', 'w') as f: json.dump({'voice': voice_name, 'files': audio_files}, f, indent=2)

total_dur = sum(a['duration'] for a in audio_files)
print(f'\n✅ {len(audio_files)} files, {total_dur:.1f}s total narration')

# Update prompts.json with actual narration durations
if prompts_file.exists():
    with open(prompts_file) as f: prompts = json.load(f)
    dur_map = {a['scene']: a['duration'] for a in audio_files}
    for p in prompts:
        if p['scene'] in dur_map:
            p['duration'] = max(3, min(10, int(dur_map[p['scene']]) + 1))  # clip duration = narration + 1s buffer
    with open(prompts_file, 'w') as f: json.dump(prompts, f, indent=2)
    print(f'✅ Updated prompts.json with narration-matched durations')

In [ ]:
# PREVIEW
for a in audio_files:
    display(Markdown(f'### Scene {a["scene"]} ({a["duration"]:.1f}s)'))
    display(Markdown(f'> {a["text"]}'))
    if Path(a['path']).exists(): display(Audio(a['path'], autoplay=False))
    display(Markdown('---'))
print('💡 Try different voices by changing CHOSEN_VOICE above!')

---
## ⏭️ Next

Audio saved to Drive. Next: **05_Generate.ipynb** (videos) then **06_Assemble.ipynb** (final).

---
*ROTBOTS Workshop — LI-MA 2026*